In [1]:
# Se importan las librerías usadas para consultar el mart analítico.

from pathlib import Path
import sqlite3

import pandas as pd
import plotly.express as px

In [2]:
# Se define la ruta del mart de referencia y los entregables de consultas.

mart_file = Path("../data/sales_mart.db")
submission_directory = Path("../submission")
submission_directory.mkdir(exist_ok=True)
assert mart_file.exists()

In [3]:
# Se inspeccionan las tablas y el grano disponible para las consultas OLAP.

with sqlite3.connect(mart_file) as connection:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", connection
    )
    fact_count = pd.read_sql_query(
        "SELECT COUNT(*) AS rows FROM fact_sales", connection
    )
tables, fact_count

(           name
 0  dim_customer
 1      dim_date
 2   dim_product
 3    fact_sales,
    rows
 0   471)

In [4]:
# Se realiza un roll-up mensual de ventas netas por región.

monthly_region_query = """
SELECT d.year, d.month, c.region, SUM(f.net_sales) AS net_sales
FROM fact_sales f
JOIN dim_date d USING(date_key)
JOIN dim_customer c USING(customer_key)
GROUP BY d.year, d.month, c.region
ORDER BY d.year, d.month, c.region
"""
with sqlite3.connect(mart_file) as connection:
    monthly_region = pd.read_sql_query(monthly_region_query, connection)
monthly_region.head()

,year,month,region,net_sales
0,2024,1,Centro,12110.0
1,2024,1,Norte,24887.0
2,2024,1,Sur,13215.5
3,2024,2,Centro,9740.0
4,2024,2,Norte,21762.0


In [5]:
# Se realiza un slice para analizar exclusivamente la región Norte por categoría.

north_category_query = """
SELECT p.category, SUM(f.net_sales) AS net_sales, SUM(f.quantity) AS units
FROM fact_sales f
JOIN dim_customer c USING(customer_key)
JOIN dim_product p USING(product_key)
WHERE c.region = 'Norte'
GROUP BY p.category
ORDER BY net_sales DESC
"""
with sqlite3.connect(mart_file) as connection:
    north_category = pd.read_sql_query(north_category_query, connection)
north_category

,category,net_sales,units
0,Servicios,92853.0,123
1,Tecnología,77188.5,132
2,Oficina,44838.5,180


In [6]:
# Se realiza un drill-down de la categoría con mayor venta hacia sus productos.

leading_category = north_category.loc[0, "category"]
product_query = """
SELECT p.product, SUM(f.net_sales) AS net_sales
FROM fact_sales f
JOIN dim_customer c USING(customer_key)
JOIN dim_product p USING(product_key)
WHERE c.region = ? AND p.category = ?
GROUP BY p.product
ORDER BY net_sales DESC
"""
with sqlite3.connect(mart_file) as connection:
    product_drilldown = pd.read_sql_query(
        product_query, connection, params=["Norte", leading_category]
    )
product_drilldown

,product,net_sales
0,Producto 9,50646.0
1,Producto 10,27265.5
2,Producto 12,9604.0
3,Producto 11,5337.5


In [7]:
# Se verifica que el roll-up mensual reconcilie con el total de ventas del mart.

with sqlite3.connect(mart_file) as connection:
    total_sales = connection.execute(
        "SELECT SUM(net_sales) FROM fact_sales"
    ).fetchone()[0]
assert monthly_region["net_sales"].sum() == total_sales
monthly_region.to_csv(submission_directory / "monthly_region_sales.csv", index=False)
north_category.to_csv(submission_directory / "north_category_sales.csv", index=False)
product_drilldown.to_csv(
    submission_directory / "north_product_drilldown.csv", index=False
)

In [8]:
# Se visualiza el roll-up mensual para comparar regiones en el mismo período.

monthly_region["month_start"] = pd.to_datetime(
    monthly_region[["year", "month"]].assign(day=1)
)
fig = px.line(
    monthly_region,
    x="month_start",
    y="net_sales",
    color="region",
    markers=True,
    title="Ventas netas mensuales por región",
    labels={"month_start": "Mes", "net_sales": "Ventas netas", "region": "Región"},
)
fig.update_layout(template="plotly_white")
fig.show()